# Strata — 2K render on Google Colab

Renders the `StrataExplainer` Remotion composition at **2560 × 1440 (2K QHD)**, 30 fps, H.264 + AAC.

**Run the Drive cell first.** It requests access to your Google Drive. Nothing is written to Drive until you approve the mount.

- **Runtime:** CPU runtime is fine (Colab GPU is not used).
- **Expected time:** roughly 1–4 hours for the full 11:52 video, depending on the Colab CPU.
- **Keep the tab open** until the final copy to Drive finishes.
- The final MP4, `render-info.json`, `storyboard.md`, render log, and last progress snapshot are saved under `MyDrive/strata-renders/strata-2k-<timestamp>/`.

The notebook pulls the latest `main` from https://github.com/lroe/strata and records the commit hash in `render-info.json`.


In [ ]:
# @title 1) Request Google Drive access
from pathlib import Path
from google.colab import drive

print("Requesting Google Drive access...")
drive.mount("/content/drive", force_remount=True)

DRIVE_ROOT = Path("/content/drive/MyDrive/strata-renders")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

probe = DRIVE_ROOT / ".strata-write-test"
probe.write_text("ok")
probe.unlink()
print(f"Google Drive access confirmed: {DRIVE_ROOT}")


In [ ]:
# @title 2) Render settings
import os
import time
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError("Run the Drive access cell first, then rerun this cell.")

REPO_URL = "https://github.com/lroe/strata.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/strata")

DRIVE_ROOT = Path("/content/drive/MyDrive/strata-renders")
RUN_ID = time.strftime("%Y%m%d-%H%M%S")
OUTPUT_DIR = DRIVE_ROOT / f"strata-2k-{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_WIDTH = 2560
TARGET_HEIGHT = 1440
SCALE = TARGET_WIDTH / 1920.0  # 1920x1080 -> 2560x1440
CONCURRENCY = 2                # Colab free: 2 vCPU. Use 1 if memory is tight.
LOCAL_OUTPUT = PROJECT_DIR / "out" / "strata-2k.mp4"
PROGRESS_FILE = PROJECT_DIR / "out" / "render-progress.json"

print(f"Target: {TARGET_WIDTH}x{TARGET_HEIGHT} @ 30fps")
print(f"Drive output: {OUTPUT_DIR}")
print(f"Concurrency: {CONCURRENCY}")


In [ ]:
%%bash
set -e

apt-get update -qq
apt-get install -y -qq ffmpeg curl gnupg ca-certificates > /tmp/apt-base.log 2>&1

# Chromium runtime libraries for Remotion's headless shell.
apt-get install -y -qq libnss3 libnspr4 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 > /tmp/apt-chrome.log 2>&1 || true
apt-get install -y -qq libasound2 > /tmp/apt-asound.log 2>&1 || apt-get install -y -qq libasound2t64 > /tmp/apt-asound2.log 2>&1 || true

NODE_MAJOR=0
if command -v node >/dev/null 2>&1; then
  NODE_MAJOR=$(node -v | sed 's/v//' | cut -d. -f1)
fi
if [ "$NODE_MAJOR" -lt 20 ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /tmp/nodesource.log 2>&1
  apt-get install -y -qq nodejs > /tmp/node-install.log 2>&1
fi

echo "Node: $(node -v)"
echo "npm:  $(npm -v)"
echo "ffmpeg: $(ffmpeg -version | head -1)"


In [ ]:
%%bash
set -e
export GIT_TERMINAL_PROMPT=0

if [ -d /content/strata/.git ]; then
  echo "Updating existing checkout..."
  git -C /content/strata fetch --depth 1 origin main
  git -C /content/strata reset --hard FETCH_HEAD
else
  echo "Cloning repository..."
  git clone --depth 1 --branch main https://github.com/lroe/strata.git /content/strata
fi

git -C /content/strata log -1 --oneline


In [ ]:
%%bash
set -e
cd /content/strata
npm ci --no-audit --no-fund
npx remotion browser ensure


## Optional 30-frame smoke test

Run this if you want to verify the 2K pipeline before committing to the full render. It renders the first second only (about 10–20 seconds).

```bash
cd /content/strata
FRAME_RANGE=0-29 MUTED=1 node scripts/render-2k.mjs out/2k-smoke.mp4 1.3333333333333333 2
```


In [ ]:
# Optional smoke test. Uncomment the next two lines to run it.
# !cd /content/strata && FRAME_RANGE=0-29 MUTED=1 node scripts/render-2k.mjs out/2k-smoke.mp4 1.3333333333333333 2


In [ ]:
# @title 3) Render 2K and stream progress
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

PROJECT_DIR = Path("/content/strata")
LOCAL_OUTPUT = PROJECT_DIR / "out" / "strata-2k.mp4"
PROGRESS_FILE = PROJECT_DIR / "out" / "render-progress.json"
LOCAL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_OUTPUT.exists():
    LOCAL_OUTPUT.unlink()
if PROGRESS_FILE.exists():
    PROGRESS_FILE.unlink()

cmd = [
    "node",
    "scripts/render-2k.mjs",
    str(LOCAL_OUTPUT),
    str(SCALE),
    str(CONCURRENCY),
]
env = os.environ.copy()
env["RENDER_PROGRESS_FILE"] = str(PROGRESS_FILE)
env["PYTHONUNBUFFERED"] = "1"

print("Starting 2K render...")
print(" ".join(cmd))
print()

log_lines = []
last_progress_print = 0.0
done_payload = None
proc = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

try:
    for raw_line in proc.stdout:
        line = raw_line.rstrip("\n")
        if not line:
            continue
        log_lines.append(line)

        if line.startswith("@@PROGRESS@@"):
            try:
                data = json.loads(line[len("@@PROGRESS@@"):])
            except json.JSONDecodeError:
                print(line, flush=True)
                continue
            now = time.time()
            if now - last_progress_print >= 2.0 or data.get("percent", 0) >= 99.9:
                print(
                    f"[{data['percent']:5.1f}%] "
                    f"{data['renderedFrames']}/{data['totalFrames']} frames | "
                    f"{data['renderFps']:.2f} fps | ETA {data['eta']} | {data['stage']}",
                    flush=True,
                )
                last_progress_print = now
        elif line.startswith("@@BUNDLE@@"):
            print(f"[bundle] {line[len('@@BUNDLE@@'):].strip()}%", flush=True)
        elif line.startswith("@@META@@"):
            meta = json.loads(line[len("@@META@@"):])
            print(f"[meta] {meta['width']}x{meta['height']} @ {meta['fps']}fps, {meta['durationInFrames']} frames", flush=True)
        elif line.startswith("@@DONE@@"):
            done_payload = json.loads(line[len("@@DONE@@"):])
            print(f"[done] {done_payload['megabytes']} MB in {done_payload['elapsedSeconds']:.0f}s", flush=True)
        elif line.startswith("@@ERROR@@"):
            print(line, flush=True)
        else:
            print(line, flush=True)
except KeyboardInterrupt:
    print("\nInterrupted — stopping the render process...", flush=True)
    proc.terminate()
    try:
        proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        proc.kill()
    (OUTPUT_DIR / "render.log").write_text("\n".join(log_lines))
    raise

return_code = proc.wait()
(OUTPUT_DIR / "render.log").write_text("\n".join(log_lines))

if return_code != 0:
    raise RuntimeError(f"Render failed with exit code {return_code}. See {OUTPUT_DIR / 'render.log'}")

if not LOCAL_OUTPUT.exists():
    raise RuntimeError("Render reported success but the MP4 is missing.")

final_path = OUTPUT_DIR / "strata-2k.mp4"
size_mb = LOCAL_OUTPUT.stat().st_size / (1024 * 1024)
print(f"Copying final video ({size_mb:.1f} MB) to Google Drive...", flush=True)
shutil.copy2(LOCAL_OUTPUT, final_path)

# Record exactly what was rendered.
def git(*args):
    return subprocess.check_output(["git", "-C", str(PROJECT_DIR), *args], text=True).strip()

commit = git("rev-parse", "HEAD")
subject = git("log", "-1", "--pretty=%s")
info = {
    "repo": "https://github.com/lroe/strata",
    "commit": commit,
    "commit_subject": subject,
    "resolution": "2560x1440",
    "fps": 30,
    "duration_in_frames": done_payload["durationInFrames"] if done_payload else 21369,
    "duration_seconds": round((done_payload["durationInFrames"] if done_payload else 21369) / 30, 3),
    "codec": "h264",
    "audio": "aac 320k",
    "file_bytes": final_path.stat().st_size,
    "render": done_payload,
    "finished_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "drive_folder": str(OUTPUT_DIR),
}
(OUTPUT_DIR / "render-info.json").write_text(json.dumps(info, indent=2))
shutil.copy2(PROJECT_DIR / "storyboard.md", OUTPUT_DIR / "storyboard.md")
if PROGRESS_FILE.exists():
    shutil.copy2(PROGRESS_FILE, OUTPUT_DIR / "render-progress.json")

print()
print("Render complete.")
print(f"Video:  {final_path}")
print(f"Info:   {OUTPUT_DIR / 'render-info.json'}")
print(f"Log:    {OUTPUT_DIR / 'render.log'}")


## Output

The final file is `strata-2k.mp4` in your Drive folder. Download it from there, or mount the folder in Colab.

If the Colab runtime disconnects before the final Drive copy, rerun the notebook from the top. The render is a single pass; it does not resume mid-render.
